# nnU-Net v2 — Lumbar Spine MRI (SPIDER)
**Expected Dice: 0.85–0.92 | ~8 hours on T4**

### Before running:
1. Settings → Accelerator → **GPU T4 x2**
2. Add SPIDER dataset
3. Run All cells in order

### What each cell does:
- **Cell 1**: Install nnU-Net + fix PyTorch for Kaggle
- **Cell 2**: Set environment variables
- **Cell 3**: Find SPIDER dataset paths
- **Cell 4**: Convert .mha → .nii.gz with label remapping
- **Cell 5**: Create dataset.json
- **Cell 6**: Preprocessing (auto patch size, spacing)
- **Cell 7**: **TRAIN** (1000 epochs, ~8h)
- **Cell 8**: Inference on val set
- **Cell 9**: Evaluate Dice per class
- **Cell 10**: Export checkpoint for server

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Install nnU-Net v2 + correct PyTorch for Kaggle T4
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os

# Kaggle default PyTorch 2.10+cu128 has torch.compile bugs
# Install 2.3.1+cu121 which is stable
# Note: torchaudio conflict warning is SAFE TO IGNORE
print('Step 1/3: Installing PyTorch 2.3.1+cu121...')
r = subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.3.1', 'torchvision==0.18.1',
    'torchaudio==2.3.1',      # match torchaudio version to avoid conflict
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], timeout=600)
print(f'  Done (exit={r.returncode})')

print('Step 2/3: Installing nnU-Net v2...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nnunetv2'], timeout=300)

print('Step 3/3: Installing dependencies...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'SimpleITK', 'batchgenerators>=0.25',
    'acvl-utils', 'dynamic-network-architectures>=0.3.1',
    'scipy', 'pandas', 'nibabel', 'dicom2nifti'
], timeout=300)

# Verify
import torch
print(f'\nPyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f'VRAM    : {vram} GB')
print('\n✓ Setup complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Configure nnU-Net Environment
# ═══════════════════════════════════════════════════════════
import os
from pathlib import Path

NNUNET_BASE  = Path('/kaggle/working/nnunet')
RAW_DIR      = NNUNET_BASE / 'raw'
PREPROC_DIR  = NNUNET_BASE / 'preprocessed'
RESULTS_DIR  = NNUNET_BASE / 'results'

for d in [RAW_DIR, PREPROC_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw']          = str(RAW_DIR)
os.environ['nnUNet_preprocessed'] = str(PREPROC_DIR)
os.environ['nnUNet_results']      = str(RESULTS_DIR)
os.environ['nnUNet_compile']      = 'false'

DATASET_ID   = 1
DATASET_NAME = f'Dataset{DATASET_ID:03d}_SpineSPIDER'
TASK_DIR     = RAW_DIR / DATASET_NAME

for sub in ['imagesTr', 'labelsTr', 'imagesTs']:
    (TASK_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'nnUNet_raw          : {RAW_DIR}')
print(f'nnUNet_preprocessed : {PREPROC_DIR}')
print(f'nnUNet_results      : {RESULTS_DIR}')
print(f'Dataset             : {DATASET_NAME}')
print('\n✓ Environment configured')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: Find SPIDER Dataset Paths
# ═══════════════════════════════════════════════════════════
import glob
from pathlib import Path
import pandas as pd

# Auto-detect overview.csv
csvs = glob.glob('/kaggle/input/**/overview.csv', recursive=True)
assert csvs, 'overview.csv not found — add SPIDER dataset via + Add Data'

OVERVIEW  = Path(csvs[0])
DATA_BASE = OVERVIEW.parent

# Find T2 images
t2s = glob.glob('/kaggle/input/**/*_t2.mha', recursive=True)
t2s = [f for f in t2s if 'SPACE' not in f]  # exclude SPACE variants
assert t2s, 'No *_t2.mha files found — check dataset path'

IMAGES_DIR = Path(t2s[0]).parent

# Find masks dir (same filenames, different folder)
all_mha = glob.glob('/kaggle/input/**/*.mha', recursive=True)
img_names = {Path(f).name for f in t2s}
mask_dirs = {
    Path(f).parent for f in all_mha
    if Path(f).name in img_names and Path(f).parent != IMAGES_DIR
}
MASKS_DIR = sorted(mask_dirs)[0] if mask_dirs else IMAGES_DIR

n_img = len(list(IMAGES_DIR.glob('*_t2.mha')))
n_msk = len(list(MASKS_DIR.glob('*_t2.mha')))

print(f'Images dir : {IMAGES_DIR}')
print(f'Masks dir  : {MASKS_DIR}')
print(f'Overview   : {OVERVIEW}')
print(f'T2 images  : {n_img}')
print(f'T2 masks   : {n_msk}')

assert n_img >= 100, f'Expected 200+ T2 images, found {n_img}'
print('\n✓ Dataset found')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: Convert SPIDER → nnU-Net Format
# T2 = channel _0000, T1 = channel _0001 (same case)
# Each patient = 1 case with 1 or 2 image channels + 1 label
# ═══════════════════════════════════════════════════════════
import SimpleITK as sitk
import numpy as np
import pandas as pd
import shutil
from pathlib import Path

# ── ALWAYS clean old files first to avoid count mismatch ──
print('Clearing old converted files...')
for sub in ['imagesTr', 'labelsTr', 'imagesTs']:
    d = TASK_DIR / sub
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
print('  Cleared imagesTr, labelsTr, imagesTs')

# SPIDER label → nnU-Net contiguous label
SPIDER_TO_NNUNET = {
    0: 0,
    1: 1, 2: 2, 3: 3, 4: 4,
    5: 5, 6: 6, 7: 7, 8: 8,
    100: 9,
    201: 10, 202: 11, 203: 12, 204: 13,
    205: 14, 206: 15, 207: 16, 208: 17,
}
NUM_CLASSES = 18

def remap_mask(mask_np):
    out = np.zeros_like(mask_np, dtype=np.uint8)
    for src, dst in SPIDER_TO_NNUNET.items():
        out[mask_np == src] = dst
    return out

def convert_cases(pids, split, out_images, out_labels):
    """Each patient = 1 case. T2=ch0, T1=ch1. Label from T2 mask."""
    ok, skip = 0, 0
    has_t1 = True  # track if T1 available
    for pid in pids:
        t2_img  = IMAGES_DIR / f'{pid}_t2.mha'
        t2_mask = MASKS_DIR  / f'{pid}_t2.mha'
        t1_img  = IMAGES_DIR / f'{pid}_t1.mha'

        if not t2_img.exists() or not t2_mask.exists():
            skip += 1; continue
        try:
            # Channel 0: T2 image
            img_t2 = sitk.ReadImage(str(t2_img))
            sitk.WriteImage(img_t2, str(out_images / f'{pid}_0000.nii.gz'))

            # Channel 1: T1 image (if exists)
            if t1_img.exists():
                img_t1 = sitk.ReadImage(str(t1_img))
                # Resample T1 to T2 space if needed
                try:
                    img_t1_res = sitk.Resample(img_t1, img_t2,
                                               sitk.Transform(),
                                               sitk.sitkLinear, 0.0,
                                               img_t2.GetPixelID())
                    sitk.WriteImage(img_t1_res, str(out_images / f'{pid}_0001.nii.gz'))
                except:
                    sitk.WriteImage(img_t1, str(out_images / f'{pid}_0001.nii.gz'))
            else:
                has_t1 = False

            # Label: from T2 mask (remap labels)
            mask_itk = sitk.ReadImage(str(t2_mask))
            mask_np  = sitk.GetArrayFromImage(mask_itk).astype(np.int32)
            mask_new = sitk.GetImageFromArray(remap_mask(mask_np))
            mask_new.CopyInformation(mask_itk)
            sitk.WriteImage(mask_new, str(out_labels / f'{pid}.nii.gz'))
            ok += 1
        except Exception as e:
            print(f'  Error {pid}: {e}')
            skip += 1

    print(f'  {split}: {ok} cases converted, {skip} skipped')
    return ok, has_t1

# Get train/val split
df = pd.read_csv(OVERVIEW)
train_pids, val_pids = [], []
seen = set()
for name in df['new_file_name'].tolist():
    if not name.endswith('_t2') or 'SPACE' in name: continue
    pid = name.replace('_t2', '')
    if pid in seen: continue
    seen.add(pid)
    sub = df.loc[df['new_file_name'] == name, 'subset'].values
    if len(sub) and sub[0] == 'validation':
        val_pids.append(pid)
    else:
        train_pids.append(pid)

print(f'Train patients: {len(train_pids)}')
print(f'Val patients  : {len(val_pids)}')
print('Converting (1 case per patient: T2=ch0, T1=ch1)...')

# Both train and val go into imagesTr (nnU-Net uses internal CV)
n_tr, has_t1_tr = convert_cases(train_pids, 'train', TASK_DIR/'imagesTr', TASK_DIR/'labelsTr')
n_va, has_t1_va = convert_cases(val_pids,   'val',   TASK_DIR/'imagesTr', TASK_DIR/'labelsTr')
HAS_T1 = has_t1_tr and has_t1_va

total_cases = len(list((TASK_DIR/'labelsTr').glob('*.nii.gz')))
total_imgs  = len(list((TASK_DIR/'imagesTr').glob('*.nii.gz')))
print(f'\nLabel files  : {total_cases}  (= num training cases)')
print(f'Image files  : {total_imgs}  (= cases x channels)')
print(f'T1 available : {HAS_T1}')
print('\n✓ Conversion complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: Create dataset.json
# numTraining = number of label files = number of patients
# ═══════════════════════════════════════════════════════════
import json
from pathlib import Path

# Count actual label files — this is the TRUE numTraining
label_files = sorted((TASK_DIR/'labelsTr').glob('*.nii.gz'))
num_training = len(label_files)
print(f'Label files (numTraining): {num_training}')
print(f'Sample case IDs: {[f.name.replace(".nii.gz","") for f in label_files[:3]]}')

# Channel names — check if enough T1 files exist
t1_files = list((TASK_DIR/'imagesTr').glob('*_0001.nii.gz'))
t2_files = list((TASK_DIR/'imagesTr').glob('*_0000.nii.gz'))
t1_coverage = len(t1_files) / max(len(t2_files), 1)
print(f'T2 files: {len(t2_files)}')
print(f'T1 files: {len(t1_files)} ({t1_coverage*100:.0f}% coverage)')

# Only use T1+T2 if ALL patients have T1 (>=95% coverage)
# Otherwise T2-only to avoid 'unexpected number of modalities' error
if t1_coverage >= 0.95:
    channel_names = {'0': 'T2', '1': 'T1'}
    print('Using T1+T2 dual channel')
else:
    channel_names = {'0': 'T2'}
    print(f'Using T2 only (T1 coverage {t1_coverage*100:.0f}% < 95%)')
    # Remove T1 files to keep folder clean
    for f in t1_files:
        f.unlink()
    print(f'  Removed {len(t1_files)} T1 files from imagesTr')

dataset_json = {
    'channel_names': channel_names,
    'labels': {
        'background'  : 0,
        'Vertebra-L5' : 1, 'Vertebra-L4' : 2, 'Vertebra-L3' : 3, 'Vertebra-L2': 4,
        'Vertebra-L1' : 5, 'Vertebra-T12': 6, 'Vertebra-T11': 7, 'Vertebra-T10': 8,
        'Sacrum'      : 9,
        'IVD-L5S1'    : 10, 'IVD-L4L5': 11, 'IVD-L3L4': 12, 'IVD-L2L3': 13,
        'IVD-L1L2'    : 14, 'IVD-T12L1': 15, 'IVD-T11T12': 16, 'IVD-T10T11': 17,
    },
    'numTraining'  : num_training,   # MUST match number of label files exactly
    'file_ending'  : '.nii.gz',
    'name'         : DATASET_NAME,
    'description'  : 'SPIDER Lumbar Spine MRI — 18 classes',
    'reference'    : 'SPIDER Dataset (CC-BY)',
    'licence'      : 'CC-BY',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}

with open(TASK_DIR / 'dataset.json', 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f'\ndataset.json saved')
print(f'  numTraining : {num_training}')
print(f'  channels    : {channel_names}')
print(f'  classes     : 18 (0=bg, 1-8=vertebrae, 9=sacrum, 10-17=IVDs)')
print('\n✓ Ready for preprocessing (Cell 6)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: nnU-Net Preprocessing
# Auto-plans: patch size, spacing, normalization, augmentation
# This can take 30-60 minutes
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os, shutil, json

# ── Step 1: Clear stale preprocessed data ───────────────────
preproc_ds = PREPROC_DIR / DATASET_NAME
if preproc_ds.exists():
    shutil.rmtree(preproc_ds)
    print(f'Cleared stale preprocessed: {preproc_ds.name}')

# ── Step 2: Remove ALL T1 files (_0001) — T2 only ────────────
t1_files = list((TASK_DIR/'imagesTr').glob('*_0001.nii.gz'))
for f in t1_files:
    f.unlink()
print(f'Removed {len(t1_files)} T1 files (T2-only mode)')

# ── Step 3: Force-write correct dataset.json ─────────────────
label_files = list((TASK_DIR/'labelsTr').glob('*.nii.gz'))
num_tr = len(label_files)
ds_json = {
    'channel_names': {'0': 'T2'},
    'labels': {
        'background': 0,
        'Vertebra-L5': 1, 'Vertebra-L4': 2, 'Vertebra-L3': 3, 'Vertebra-L2': 4,
        'Vertebra-L1': 5, 'Vertebra-T12': 6, 'Vertebra-T11': 7, 'Vertebra-T10': 8,
        'Sacrum': 9,
        'IVD-L5S1': 10, 'IVD-L4L5': 11, 'IVD-L3L4': 12, 'IVD-L2L3': 13,
        'IVD-L1L2': 14, 'IVD-T12L1': 15, 'IVD-T11T12': 16, 'IVD-T10T11': 17,
    },
    'numTraining': num_tr,
    'file_ending': '.nii.gz',
    'name': DATASET_NAME,
    'description': 'SPIDER Lumbar Spine MRI T2',
    'licence': 'CC-BY',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(TASK_DIR / 'dataset.json', 'w') as f:
    json.dump(ds_json, f, indent=2)
print(f'dataset.json  : numTraining={num_tr}, channels=T2-only')

# ── Step 4: Final sanity check ───────────────────────────────
t2_count = len(list((TASK_DIR/'imagesTr').glob('*_0000.nii.gz')))
t1_left  = len(list((TASK_DIR/'imagesTr').glob('*_0001.nii.gz')))
print(f'T2 images     : {t2_count}')
print(f'Label files   : {num_tr}')
print(f'T1 files left : {t1_left}  (must be 0)')
assert t1_left == 0, 'ERROR: T1 files still present — re-run this cell'
assert t2_count == num_tr, f'ERROR: {t2_count} T2 images vs {num_tr} labels'
print('All checks passed!')
print()

env = {
    **os.environ,
    'nnUNet_raw'         : str(RAW_DIR),
    'nnUNet_preprocessed': str(PREPROC_DIR),
    'nnUNet_results'     : str(RESULTS_DIR),
    'nnUNet_compile'     : 'false',
}

print('Running nnUNetv2_plan_and_preprocess...')
print('This auto-determines optimal patch size, spacing, normalization.')
print('Expected time: 20-60 minutes for SPIDER dataset.')
print('='*55)

# Use CLI — more reliable than Python module on Kaggle
import shutil
cli = shutil.which('nnUNetv2_plan_and_preprocess') or '/usr/local/bin/nnUNetv2_plan_and_preprocess'

result = subprocess.run(
    [sys.executable, cli,
     '-d', str(DATASET_ID),
     '-c', '2d',
     '--verify_dataset_integrity',
     '-np', '2'],
    env=env, timeout=7200, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

output = result.stdout or ''
# Print last 2000 chars (most informative)
print(output[-2000:] if len(output) > 2000 else output)

if result.returncode != 0:
    print(f'\nPreprocessing exit code: {result.returncode}')
    print('Check output above for errors.')
    print('Common fix: re-run Cell 4 and Cell 5 to ensure files are correct.')
else:
    print('\n✓ Preprocessing complete')
    # Show what was planned
    plans_file = PREPROC_DIR / DATASET_NAME / 'nnUNetPlans.json'
    if plans_file.exists():
        import json
        plans = json.loads(plans_file.read_text())
        cfg_2d = plans.get('configurations', {}).get('2d', {})
        print(f'  Patch size  : {cfg_2d.get("patch_size")}')
        print(f'  Batch size  : {cfg_2d.get("batch_size")}')
        print(f'  Spacing     : {cfg_2d.get("spacing")}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: TRAIN nnU-Net v2 — 2D config, fold 0
# FIXES:
#   1. Patches torch.compile for Python 3.12 (Kaggle)
#   2. Uses sitecustomize.py for subprocess safety
#   3. Streams output so you see epoch progress
# Expected: epoch 1 in ~60s, Dice >0.85 by epoch ~600
# Total time: ~8 hours on T4
# ═══════════════════════════════════════════════════════════
import os, sys, subprocess, shutil, site
from pathlib import Path

# ── Set env vars ────────────────────────────────────────────
env = {
    **os.environ,
    'nnUNet_raw'         : str(RAW_DIR),
    'nnUNet_preprocessed': str(PREPROC_DIR),
    'nnUNet_results'     : str(RESULTS_DIR),
    'nnUNet_compile'     : 'false',
    'PYTHONDONTWRITEBYTECODE': '1',
}

# ── Patch torch.compile in nnUNetTrainer.py ─────────────────
import importlib.util
spec = importlib.util.find_spec('nnunetv2')
if spec:
    pkg = Path(spec.origin).parent
    tf  = pkg / 'training' / 'nnUNetTrainer' / 'nnUNetTrainer.py'
    if tf.exists():
        src = tf.read_text()
        patched = False
        for old, new in [
            ('self.network = torch.compile(self.network)',
             'self.network = self.network  # compile disabled Py3.12'),
            ("self.compile = ('nnUNet_compile' in os.environ "
             "and os.environ['nnUNet_compile'].lower() in ('true', '1', 't'))",
             'self.compile = False  # compile disabled Py3.12'),
        ]:
            if old in src:
                src = src.replace(old, new); patched = True
                print(f'  Patched: {old[:50]}...')
        if patched:
            tf.write_text(src)
            print('  Trainer file saved.')
        else:
            if 'compile disabled' in src:
                print('  Already patched.')
            else:
                print('  Pattern not found — may be different nnU-Net version (ok)')

# ── sitecustomize.py safety net ─────────────────────────────
sc = Path(site.getsitepackages()[0]) / 'sitecustomize.py'
sc.write_text("""
try:
    import torch as _t
    if not getattr(_t, '_cpatch', False):
        _t.compile = lambda fn=None, *a, **kw: fn if fn else (lambda f: f)
        _t._cpatch = True
except: pass
""")
print(f'  sitecustomize.py written')

# ── Verify preprocessing is done ───────────────────────────
preproc_check = PREPROC_DIR / DATASET_NAME / 'nnUNetPlans.json'
if not preproc_check.exists():
    print('\nERROR: Preprocessing not done. Run Cell 6 first!')
    raise RuntimeError('Run Cell 6 first')
print(f'  Preprocessing verified: {preproc_check}')

# ── Find nnUNetv2_train CLI ─────────────────────────────────
cli = shutil.which('nnUNetv2_train') or '/usr/local/bin/nnUNetv2_train'
assert Path(cli).exists(), f'nnUNetv2_train not found at {cli}'
print(f'  CLI: {cli}')

# ── Launch training ─────────────────────────────────────────
cmd = [sys.executable, cli, str(DATASET_ID), '2d', '0', '--npz']
print(f'\nCommand: {" ".join(cmd)}')
print('Starting training — streaming output below')
print('Expected: Dice >0.85 around epoch 600-800')
print('='*55)

# Set num_workers=0 to prevent data loader hanging on Kaggle
env['nnUNet_n_proc_DA'] = '0'

proc = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\nInterrupted — checkpoint saved. Re-run this cell to resume.')

proc.wait()
rc = proc.returncode
print(f'\nReturn code: {rc}')
if rc == 0:
    print('Training complete! Run Cell 8 for inference.')
else:
    print('Non-zero exit — check output above.')
    print('If it shows checkpoint progress, re-run this cell (auto-resume).')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8: Run Inference on Validation Set
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os, shutil
from pathlib import Path

PRED_DIR = Path('/kaggle/working/predictions')
PRED_DIR.mkdir(exist_ok=True)

env = {
    **os.environ,
    'nnUNet_raw': str(RAW_DIR),
    'nnUNet_preprocessed': str(PREPROC_DIR),
    'nnUNet_results': str(RESULTS_DIR),
}

cli = shutil.which('nnUNetv2_predict') or '/usr/local/bin/nnUNetv2_predict'

print('Running inference on validation images...')
result = subprocess.run([
    sys.executable, cli,
    '-i', str(TASK_DIR / 'imagesTr'),
    '-o', str(PRED_DIR),
    '-d', str(DATASET_ID),
    '-c', '2d',
    '-f', '0',
    '--save_probabilities',
], env=env, text=True, timeout=7200,
   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print(result.stdout[-1000:] if result.stdout else 'No output')
print(f'Exit code: {result.returncode}')
print(f'\nPredictions saved to: {PRED_DIR}')
print(f'Files: {len(list(PRED_DIR.glob("*.nii.gz")))}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9: Evaluate — Dice + IoU per class
# ═══════════════════════════════════════════════════════════
import numpy as np
import SimpleITK as sitk
from pathlib import Path
from collections import defaultdict
import json

LABEL_NAMES = {
    1:'Vert-L5', 2:'Vert-L4', 3:'Vert-L3', 4:'Vert-L2',
    5:'Vert-L1', 6:'Vert-T12', 7:'Vert-T11', 8:'Vert-T10',
    9:'Sacrum',
    10:'IVD-L5S1', 11:'IVD-L4L5', 12:'IVD-L3L4', 13:'IVD-L2L3',
    14:'IVD-L1L2', 15:'IVD-T12L1', 16:'IVD-T11T12', 17:'IVD-T10T11',
}

pred_files = sorted(Path('/kaggle/working/predictions').glob('*.nii.gz'))
gt_dir     = TASK_DIR / 'labelsTr'
print(f'Evaluating {min(len(pred_files),30)} cases...')

all_dice = defaultdict(list)
all_iou  = defaultdict(list)
sm = 1e-6

for pred_path in pred_files[:30]:
    gt_path = gt_dir / pred_path.name
    if not gt_path.exists(): continue
    pred = sitk.GetArrayFromImage(sitk.ReadImage(str(pred_path)))
    gt   = sitk.GetArrayFromImage(sitk.ReadImage(str(gt_path)))
    for label, name in LABEL_NAMES.items():
        p = (pred == label).astype(float); g = (gt == label).astype(float)
        if g.sum() == 0 and p.sum() == 0: continue
        tp = (p*g).sum()
        all_dice[name].append(float(2*tp / (p.sum()+g.sum()+sm)))
        all_iou[name].append(float(tp / (p.sum()+g.sum()-tp+sm)))

print(f'\n{"Class":<14} {"Dice":>8}  {"IoU":>8}  Grade')
print('-'*40)
all_d, vert_d, ivd_d = [], [], []
per_class = {}
for name in LABEL_NAMES.values():
    if name not in all_dice: continue
    d  = float(np.mean(all_dice[name]))
    io = float(np.mean(all_iou[name]))
    all_d.append(d); per_class[name] = d
    if 'Vert' in name: vert_d.append(d)
    if 'IVD'  in name: ivd_d.append(d)
    g = 'EXCELLENT' if d>=0.90 else 'GOOD' if d>=0.80 else 'OK' if d>=0.70 else 'POOR'
    print(f'{name:<14} {d:>8.4f}  {io:>8.4f}  {g}')

md = float(np.mean(all_d)) if all_d else 0
vd = float(np.mean(vert_d)) if vert_d else 0
id_ = float(np.mean(ivd_d)) if ivd_d else 0

print('='*40)
print(f'Vertebrae mean Dice : {vd:.4f}')
print(f'IVD mean Dice       : {id_:.4f}')
print(f'Overall mean Dice   : {md:.4f}')
print('='*40)

if   md >= 0.90: print('TARGET ACHIEVED: Dice >= 0.90!')
elif md >= 0.85: print('GOAL MET: Dice >= 0.85!')
elif md >= 0.75: print('Good — close to target')
else:            print('Keep training — more epochs needed')

with open('/kaggle/working/nnunet_eval.json', 'w') as f:
    json.dump({'mean_dice':md,'vertebrae':vd,'ivd':id_,'per_class':per_class}, f, indent=2)
print('\nResults saved to /kaggle/working/nnunet_eval.json')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10: Export checkpoint
# Saves best_model.pth in nnU-Net format
# Download from Kaggle Output tab → copy to outputs/gpu_run/
# Then run: python scripts/convert_nnunet_ckpt.py
# ═══════════════════════════════════════════════════════════
import shutil, os, json
from pathlib import Path

model_dir = Path(RESULTS_DIR) / DATASET_NAME / 'nnUNetTrainer__nnUNetPlans__2d' / 'fold_0'
print(f'Model dir: {model_dir}')
print(f'Exists   : {model_dir.exists()}')

if model_dir.exists():
    ckpts = list(model_dir.glob('*.pth'))
    print(f'Checkpoints: {[c.name for c in ckpts]}')

    best = model_dir / 'checkpoint_best.pth'
    last = model_dir / 'checkpoint_final.pth'

    if best.exists():
        shutil.copy(best, '/kaggle/working/nnunet_best.pth')
        print('Saved: /kaggle/working/nnunet_best.pth')
    elif last.exists():
        shutil.copy(last, '/kaggle/working/nnunet_best.pth')
        print('Saved final checkpoint: /kaggle/working/nnunet_best.pth')

    # Also save full model folder as ZIP
    shutil.make_archive('/kaggle/working/nnunet_fold0', 'zip', str(model_dir))
    print('Archived: /kaggle/working/nnunet_fold0.zip')

    # Save plans.json for inference later
    plans_src = PREPROC_DIR / DATASET_NAME / 'nnUNetPlans.json'
    if plans_src.exists():
        shutil.copy(plans_src, '/kaggle/working/nnUNetPlans.json')
        print('Saved: /kaggle/working/nnUNetPlans.json')

    # Save evaluation results too
    eval_f = Path('/kaggle/working/nnunet_eval.json')
    if eval_f.exists():
        ev = json.loads(eval_f.read_text())
        print(f'\nFinal Dice: {ev.get("mean_dice",0):.4f}')

    print('\n=== DOWNLOAD INSTRUCTIONS ===')
    print('1. Go to Kaggle Output tab (right panel)')
    print('2. Download: nnunet_best.pth  (checkpoint)')
    print('             nnUNetPlans.json (inference config)')
    print('             nnunet_fold0.zip (full model)')
    print('3. Copy nnunet_best.pth to:')
    print('   c:\\project\\Spine Segmentation\\ATM-Net++\\outputs\\gpu_run\\nnunet_best.pth')
    print('4. Copy nnUNetPlans.json to same folder')
    print('5. Server will auto-load it (highest dice wins)')
else:
    print('Model dir not found — training may not have finished')
    # Search everywhere
    found = list(Path(RESULTS_DIR).glob('**/*.pth'))
    print(f'All .pth files: {found[:5]}')